In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

<b>MODEL CALL</b>

In [ ]:
from google import genai

client = genai.Client()
model = "gemini-2.5-flash"

<b>Structured Output</b>

In [ ]:
from typing import List, Literal
from pydantic import BaseModel, Field


class AuditCoAuthorOutput(BaseModel):
    task_type: Literal[
        "full_audit_and_draft",
        "audit_only",
        "draft_only",
        "general_help",
    ] = Field(
        description="What kind of help the user is asking for."
    )

    trust_score: str = Field(
        description="A plain-language score from 1-10 with a label (e.g., '3/10 — Low Trust')."
    )

    weaknesses_flagged: List[str] = Field(
        description="Specific methodological flaws found in the research, explained simply."
    )

    audit_reasoning: str = Field(
        description="A short, beginner-friendly paragraph summarizing why the paper received its trust score."
    )

    document_type: str = Field(
        description="The type of document being co-authored (e.g., Peer Review Critique, Literature Review)."
    )

    section_content: List[str] = Field(
        description="The drafted paragraphs or bullet points for the critique document."
    )

    writing_tips: List[str] = Field(
        description="Specific, actionable tips to improve clarity or tone when discussing these weaknesses."
    )

    follow_up_actions: List[str] = Field(
        description="Next steps the user should take (e.g., checking full paper, looking for replications)."
    )

<b>System Prompt</b>

In [ ]:
import yaml
from pathlib import Path

_PROMPTS = yaml.safe_load(Path("prompt.yml").read_text(encoding="utf-8"))
TRUST_AUDIT_SYSTEM = _PROMPTS["TRUST_AUDIT_COAUTHOR_SYSTEM"]

<b>Cost & Pricing</b>

In [ ]:
IN_PRICE_PER_MTOK = 0.15
OUT_PRICE_PER_MTOK = 0.60

totals = {"input": 0, "output": 0}


def track_cost(usage) -> None:
    totals["input"] += usage.prompt_token_count or 0
    totals["output"] += usage.candidates_token_count or 0


def usage_report() -> str:
    """Summarise tokens used and the estimated cost so far."""
    cost = (
        totals["input"] / 1e6 * IN_PRICE_PER_MTOK
        + totals["output"] / 1e6 * OUT_PRICE_PER_MTOK
    )
    return f"{totals['input']} in + {totals['output']} out tokens = ~${cost:.4f}"

<b>Trust-Audit Co-Author Agent</b>

In [ ]:
def trust_audit_agent(message: str) -> AuditCoAuthorOutput:
    resp = client.models.generate_content(
        model=model,
        contents=message,
        config={
            "system_instruction": TRUST_AUDIT_SYSTEM,
            "response_mime_type": "application/json",
            "response_schema": AuditCoAuthorOutput,
        },
    )
    track_cost(resp.usage_metadata)
    return resp.parsed

<b>Draft Reply</b>

In [ ]:
def draft_reply(result: AuditCoAuthorOutput) -> None:
    print("TASK TYPE       :", result.task_type)
    print("TRUST SCORE     :", result.trust_score)
    print("DOCUMENT TYPE   :", result.document_type)

    print("\nWEAKNESSES FLAGGED:")
    if result.weaknesses_flagged:
        for weakness in result.weaknesses_flagged:
            print(f"  - {weakness}")
    else:
        print("  - None detected.")

    print("\nAUDIT REASONING:")
    print(f"  {result.audit_reasoning}")

    print("\nDRAFTED CRITIQUE:")
    if result.section_content:
        for point in result.section_content:
            print(f"  - {point}")
    else:
        print("  - None")

    print("\nWRITING TIPS:")
    if result.writing_tips:
        for tip in result.writing_tips:
            print(f"  - {tip}")
    else:
        print("  - None")

    print("\nFOLLOW-UP ACTIONS:")
    if result.follow_up_actions:
        for action in result.follow_up_actions:
            print(f"  - {action}")
    else:
        print("  - None")

<b>Run Agent</b>

In [ ]:
def main() -> None:
    samples = [
        "Audit this abstract and draft a critique for my lit review: 'A study of 12 participants showed that drinking green tea daily reduced anxiety by 40%. The study was funded by GreenTea Corp and had no control group. Participants were self-selected volunteers from a wellness forum.'",
    ]

    for s in samples:
        result = trust_audit_agent(s)

        print("-" * 80)
        print("MESSAGE :", s)
        print("-" * 80)
        draft_reply(result)
        print()

    print("=" * 80)
    print("TOKENS  :", usage_report())


if __name__ == "__main__":
    main()